# 3.1 — Runoff accuracy assessment at the annual basin scale

The analysis compares annual CONUS404 runoff with annual USGS runoff for the Sanford/CAMELS basin set. The aligned input table contains one modeled and observed runoff depth in millimeters for each basin and year. The workflow retains finite, positive pairs inside the configured study period and calculates the metrics independently for each basin.

For basin $b$, the analysis calculates MBE, MAE, RMSE, and PBIAS across its valid annual pairs. The analysis normalizes basin MAE by the basin's multiannual mean modeled runoff:

$$
\mathrm{nMAE}_b(\%)=100\frac{\frac{1}{n_b}\sum_t|Q_{b,t}^{\mathrm{CONUS404}}-Q_{b,t}^{\mathrm{USGS}}|}{\overline{Q}_{b}^{\mathrm{CONUS404}}}.
$$

The workflow uses the modeled denominator consistently with the precipitation and evapotranspiration assessments. The workflow retains only basins with at least the configured number of valid annual pairs.


## Basin metrics and climate-region assignment

The workflow assigns each gage point to a climate-region polygon with a spatial within-join. It merges the region identity with the basin metrics and preserves the number of annual pairs used for each result.


In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import pandas as pd

REPOSITORY_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPOSITORY_ROOT / "src"))

from accuracy_assessment import load_config
from regional_plots import plot_table_violins
from runoff_assessment import assign_climate_regions, compute_basin_metrics

config = load_config(REPOSITORY_ROOT / "config.yml")
runoff_config = config["runoff"]
period = config["study"]["accuracy"]["runoff"]["annual"]
region_config = config["regions"]
regions = gpd.read_file(region_config["file"])
output_dir = Path(config["output_dir"]) / "accuracy" / "runoff" / "annual"
output_dir.mkdir(parents=True, exist_ok=True)

aligned = pd.read_csv(runoff_config["input_csv"])
metrics = compute_basin_metrics(
    aligned,
    runoff_config["columns"],
    int(period["start_year"]),
    int(period["end_year"]),
    int(runoff_config["minimum_years"]),
)
metrics_by_region = assign_climate_regions(
    metrics,
    runoff_config["gage_points_file"],
    runoff_config["gage_id_field"],
    regions,
    region_config["name_field"],
    region_config["code_field"],
)
metrics_path = output_dir / "runoff_accuracy_by_basin.csv"
metrics_by_region.to_csv(metrics_path, index=False)
metrics_by_region


## Basin nMAE distributions

The analysis groups basin nMAE values by climate region. Each basin contributes one multiannual nMAE value, so the violin distributions represent between-basin variation rather than raster-cell variation.


In [ ]:
figure_config = config["figures"]
runoff_figure_config = figure_config["accuracy"]["runoff"]
nmae_stats = plot_table_violins(
    metrics_by_region,
    [("CONUS404 vs USGS", "nmae_percent")],
    region_config["name_field"],
    region_config["code_field"],
    Path(config["output_dir"]) / "accuracy" / "runoff" / "figures" / "annual_nmae_violin.png",
    Path(config["output_dir"]) / "accuracy" / "runoff" / "tables" / "annual_nmae_regional_statistics.csv",
    "nMAE (%)",
    tuple(runoff_figure_config["nmae_range"]),
    [figure_config["colors"][2]],
    dpi=figure_config["dpi"],
)
nmae_stats


## Basin PBIAS distributions

The analysis groups the basin PBIAS values by the same climate regions. Positive values indicated runoff overestimation by CONUS404 and negative values indicated underestimation relative to USGS.


In [ ]:
pbias_stats = plot_table_violins(
    metrics_by_region,
    [("CONUS404 vs USGS", "pbias_percent")],
    region_config["name_field"],
    region_config["code_field"],
    Path(config["output_dir"]) / "accuracy" / "runoff" / "figures" / "annual_pbias_violin.png",
    Path(config["output_dir"]) / "accuracy" / "runoff" / "tables" / "annual_pbias_regional_statistics.csv",
    "PBIAS (%)",
    tuple(runoff_figure_config["pbias_range"]),
    [figure_config["colors"][3]],
    dpi=figure_config["dpi"],
)
pbias_stats
